In [1]:
import pandas as pd
import os
import sys


sys.path.append(os.path.dirname(os.path.abspath('__file__')))

from feature import df

df['hour_sin'] = pd.to_numeric(df['hour_sin'], errors='coerce')
df = df.dropna()

print(df.shape)
print(df.dtypes)


   hour_sin  hour_cos  soil_temp  rain_fall  prev_5  prev_4  prev_3  prev_2  \
0    1.0000    0.0000       14.1        0.0    41.6    41.6    41.6    41.6   
1    0.9659   -0.2588       14.2        0.0    41.6    41.6    41.6    41.6   
2    0.8660   -0.5000       14.2        0.5    41.6    41.6    41.6    41.6   
3    0.7071   -0.7071       14.3        4.1    41.6    41.6    41.6    41.6   
4    0.5000   -0.8660       14.2        2.2    41.6    41.6    41.6    41.6   

   prev_1  future_moisture  
0    41.6             41.6  
1    41.6             41.6  
2    41.6             41.7  
3    41.6             42.6  
4    41.7             42.5  
(3814, 10)
hour_sin           float64
hour_cos           float64
soil_temp          float64
rain_fall          float64
prev_5             float64
prev_4             float64
prev_3             float64
prev_2             float64
prev_1             float64
future_moisture    float64
dtype: object


In [2]:
df.corr()['future_moisture']

hour_sin          -0.004466
hour_cos          -0.008164
soil_temp         -0.582910
rain_fall          0.097935
prev_5             0.999036
prev_4             0.999224
prev_3             0.999410
prev_2             0.999587
prev_1             0.999761
future_moisture    1.000000
Name: future_moisture, dtype: float64

In [7]:
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

x = df.drop(columns='future_moisture')
y = df['future_moisture']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

scalar = StandardScaler()

x_train = scalar.fit_transform(x_train)
x_test = scalar.transform(x_test)


lr = LinearRegression()
sgd = SGDRegressor(
    loss='squared_error',
    penalty=None,
    learning_rate='constant',
    eta0=0.01,
    max_iter=50000,
    tol=1e-8,
    random_state=42
)
rf = RandomForestRegressor(max_depth= 6, random_state=42)

lr.fit(x_train, y_train)
sgd.fit(x_train, y_train)
rf.fit(x_train, y_train)

models = {
     "linear": lr,
    'forest': rf,
    'sgd': sgd
}

results = []

for name,model in models.items():
    
    y_pred = model.predict(x_test)

    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    
    results.append({
        'Model': name,
        'R2 Score': r2,
        'RMSE': rmse
    })

result_df = pd.DataFrame(results)
print(f"Converged: {sgd.n_iter_} iterations")
print(result_df)


df.to_csv('soil_moist.csv')

Converged: 8 iterations
    Model  R2 Score      RMSE
0  linear  0.952178  0.082072
1  forest -5.121324  0.928546
2     sgd  0.903954  0.116311


In [4]:
from sklearn.model_selection import cross_val_score

eta_values = [0.1, 0.05, 0.01, 0.005, 0.001, 0.0005]

for eta in eta_values:
    sgd_test = SGDRegressor(
        loss='squared_error',
        penalty=None,
        learning_rate='constant',
        eta0=eta,
        max_iter=50000,
        tol=1e-8,
        random_state=42
    )
    sgd_test.fit(x_train, y_train)
    y_pred = sgd_test.predict(x_test)
    r2 = r2_score(y_test, y_pred)
    print(f"eta0={eta:<8} | n_iter={sgd_test.n_iter_:<6} | R²={r2:.6f}")

eta0=0.1      | n_iter=6      | R²=-10478032541373851648.000000
eta0=0.05     | n_iter=13     | R²=-2307.748570
eta0=0.01     | n_iter=8      | R²=0.903954
eta0=0.005    | n_iter=8      | R²=0.889940
eta0=0.001    | n_iter=13     | R²=0.881196
eta0=0.0005   | n_iter=13     | R²=0.849421


In [6]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error

X = df.drop(columns=['future_moisture'])
y = df['future_moisture']

x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=0.3, shuffle=False)

model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
print(r2_score(y_test, y_pred))
print(root_mean_squared_error(y_test, y_pred))

-0.4925593512843347
12.241004850406346
